In [4]:
import numpy as np
import pandas as pd
import math

#Charger les données

# Charger le dataset depuis le fichier CSV
df = pd.read_csv("data.csv")

# Supprimer la colonne vide si elle existe
if 'Unnamed: 32' in df.columns:
    df = df.drop(columns=['Unnamed: 32'])

# Supprimer la colonne id (inutile pour le clustering)
if 'id' in df.columns:
    df = df.drop(columns=['id'])

# On garde uniquement les variables explicatives (pas la cible)
X = df.drop(columns=['diagnosis']).values.astype(float)


#  Normalisation

# Mettre toutes les variables entre 0 et 1
# Important car DBSCAN utilise des distances
X = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))


#  Fonction distance

# Calcul de la distance euclidienne entre deux points
def distance(a, b):
    return math.sqrt(np.sum((a - b) ** 2))


#  Trouver les voisins d’un point

def get_neighbors(X, point_idx, eps):
    neighbors = []  # liste des voisins

    # Parcourir tous les points du dataset
    for i in range(len(X)):
        # Si la distance est inférieure au rayon eps
        if distance(X[point_idx], X[i]) <= eps:
            neighbors.append(i)  # ajouter comme voisin

    return neighbors




In [5]:
#  Algorithme DBSCAN 

def dbscan(X, eps=0.3, min_samples=5):

    n = len(X)  # nombre total de points

    # Initialisation :
    # -1 signifie bruit (noise)
    labels = [-1] * n

    cluster_id = 0  # identifiant des clusters

    # Liste pour savoir si un point a déjà été visité
    visited = [False] * n

    # Parcourir tous les points
    for i in range(n):

        # Si déjà traité, on passe au suivant
        if visited[i]:
            continue

        # Marquer comme visité
        visited[i] = True

        # Trouver les voisins du point
        neighbors = get_neighbors(X, i, eps)

        # Si pas assez de voisins → bruit
        if len(neighbors) < min_samples:
            labels[i] = -1

        else:
            # Nouveau cluster
            labels[i] = cluster_id

            # Copier les voisins dans une file (queue)
            queue = neighbors.copy()

            # Explorer tous les voisins
            while queue:
                j = queue.pop(0)

                # Si le point n’a pas encore été visité
                if not visited[j]:
                    visited[j] = True

                    # Trouver ses voisins
                    new_neighbors = get_neighbors(X, j, eps)

                    # Si c’est un point cœur → étendre le cluster
                    if len(new_neighbors) >= min_samples:
                        queue += new_neighbors

                # Si le point est encore marqué comme bruit
                # on l’ajoute au cluster
                if labels[j] == -1:
                    labels[j] = cluster_id

            # Passer au cluster suivant
            cluster_id += 1

    return labels




In [6]:
# Appliquer DBSCAN

# Lancer l'algorithme avec eps et min_samples
labels = dbscan(X, eps=0.3, min_samples=5)


# Afficher les résultats


# Afficher les clusters trouvés
print("Clusters trouvés :", set(labels))

# Nombre de clusters (sans compter le bruit -1)
print("Nombre de clusters :", len(set(labels)) - (1 if -1 in labels else 0))



Clusters trouvés : {0, 1, 2, 3, -1}
Nombre de clusters : 4
